# entrenamiento.ipynb — Hackathon NN

Pipeline completo: **carga → split → barrido de modelos → ensemble**.

**El sábado**: ajustar la celda marcada con `# ── AJUSTAR AQUÍ ──` y ejecutar *Run All*.

Tiempo estimado (CPU, 4 modelos × 300 épocas + ensemble 5 seeds): ~15-25 min.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import warnings
warnings.simplefilter('ignore')

from utils import (
    create_time_series_data, make_splits, load_data,
    build_model, train_model, train_ensemble,
    evaluate, eval_mae_naive,
    build_results_df, plot_history, apply_ffd,
    INPUT_WINDOWS, OUTPUT_WINDOWS,
)
print('utils OK')

## 1. Cargar datos

**Ajustar esta celda cuando lleguen los datos del hackathon.**

In [ ]:
# ── AJUSTAR AQUÍ ─────────────────────────────────────────────────────────────
FILEPATH    = 'data/precios.csv'  # ruta al CSV / parquet del hackathon
PRICE_COLS  = None                # None = todas las numéricas como precios
                                  # o ['Close', 'Adj Close', ...]
RETURN_COLS = None                # si el CSV ya trae retornos: ['ret_SPY', ...]
FFD_D       = None                # None = retornos crudos
                                  # 0.2  = FFD óptimo (SOLO si V_OUT = 1)
# ─────────────────────────────────────────────────────────────────────────────

X_src, y_src = load_data(FILEPATH, price_cols=PRICE_COLS,
                          return_cols=RETURN_COLS, ffd_d=FFD_D)
display(X_src.head(3))

## 2. Ventanas deslizantes + split temporal

Ajustar `V_IN` y `V_OUT` según el target del hackathon.

In [ ]:
# ── AJUSTAR AQUÍ ─────────────────────────────────────────────────────────────
V_IN   = 10    # días de historia de entrada
V_OUT  = 1     # días de horizonte de predicción
EPOCHS = 300   # épocas de entrenamiento (bajar a 50 para prueba rápida)
# ─────────────────────────────────────────────────────────────────────────────

X, _  = create_time_series_data(X_src, V_IN, V_OUT)
_, y  = create_time_series_data(y_src, V_IN, V_OUT)
X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

N_FEAT    = X.shape[2]    # número de features / activos de entrada
N_TARGETS = y.shape[1]    # dimensión del target

print(f'X: {X.shape}   y: {y.shape}')
print(f'Train: {X_tr.shape[0]}  Val: {X_v.shape[0]}  Test: {X_ts.shape[0]}')
print(f'n_features={N_FEAT}  n_targets={N_TARGETS}')
print(f'\nBaseline naive  MAE test = {eval_mae_naive(X_ts, y_ts):.4f}')

## 3. Prueba rápida — un modelo

Verificar que el pipeline funciona antes del barrido completo (50 épocas).

In [ ]:
TIPO_RAPIDO = 'lstm'  # cambiar si se quiere probar otro primero

model = build_model(TIPO_RAPIDO, V_IN, n_features=N_FEAT, n_targets=N_TARGETS)
hist  = train_model(model, X_tr, y_tr, X_v, y_v, epochs=50)
res   = evaluate(model, X_tr, X_v, X_ts, y_tr, y_v, y_ts)

print(f'MAE   train={res["train"]:.4f}  val={res["val"]:.4f}  test={res["test"]:.4f}')
print(f'RMSE  test={res["rmse"]:.4f}  |  Dir acc test={res["dir"]:.2%}')
plot_history(hist, title=f'{TIPO_RAPIDO} (50 épocas prueba)')

## 4. Barrido completo — 4 modelos

Entrena las 4 arquitecturas base con sus learning rates óptimos.

In [ ]:
# LR validado en tarea previa: dense necesita lr menor para no converger demasiado rápido
LR_MAP = {'dense': 1e-4, 'lstm': 3e-4, 'cnn1d': 3e-4, 'cnn_lstm': 3e-4}

results = {}
for tipo in ['dense', 'lstm', 'cnn1d', 'cnn_lstm']:
    print(f'\n══ {tipo.upper()} ══')
    model = build_model(tipo, V_IN, n_features=N_FEAT,
                        n_targets=N_TARGETS, lr=LR_MAP[tipo])
    hist  = train_model(model, X_tr, y_tr, X_v, y_v, epochs=EPOCHS)
    res   = evaluate(model, X_tr, X_v, X_ts, y_tr, y_v, y_ts)
    results[(tipo, V_IN, V_OUT)] = res
    print(f'  MAE test={res["test"]:.4f}  dir={res["dir"]:.2%}  params={res["params"]:,}')
    plot_history(hist, title=tipo)

# Tabla comparativa
df_res = build_results_df(results)
print('\n── Tabla comparativa ──')
print(df_res[['train', 'val', 'test']].sort_values('test').to_string())

mejor_tipo = df_res['test'].idxmin()[0]
print(f'\nMejor modelo en test: {mejor_tipo}  '
      f'MAE={df_res["test"].min():.4f}')

## 5. Ensemble del mejor modelo — palanca principal

Promediar N semillas del mismo modelo elimina el ruido de inicialización
y reduce la varianza del MAE en test. Primera palanca en competición.

In [ ]:
TIPO_ENSEMBLE = mejor_tipo  # o fijar manualmente: 'lstm', 'cnn1d', etc.
N_SEEDS = 5                 # subir a 10 si sobra tiempo

print(f'Ensemble: {N_SEEDS} semillas de {TIPO_ENSEMBLE}  '
      f'({EPOCHS} épocas cada una)\n')
ens = train_ensemble(
    TIPO_ENSEMBLE, X_tr, y_tr, X_v, y_v, X_ts, y_ts,
    V_in=V_IN, n_features=N_FEAT, n_targets=N_TARGETS,
    n_seeds=N_SEEDS, epochs=EPOCHS, lr=LR_MAP[TIPO_ENSEMBLE],
)

## 6. Resumen final

In [ ]:
naive_mae = eval_mae_naive(X_ts, y_ts)
best_mae  = df_res['test'].min()
ens_mae   = ens['mae_test']

summary = pd.DataFrame({
    'Modelo':       ['naive', f'mejor_simple ({mejor_tipo})', f'ensemble_{N_SEEDS}seeds ({TIPO_ENSEMBLE})'],
    'MAE test':     [naive_mae, best_mae, ens_mae],
    'vs naive':     [0, (best_mae - naive_mae) / naive_mae,
                       (ens_mae  - naive_mae) / naive_mae],
    'vs best_simple': ['-', 0, (ens_mae - best_mae) / best_mae],
})
summary['MAE test'] = summary['MAE test'].map('{:.4f}'.format)
summary['vs naive'] = summary['vs naive'].apply(
    lambda x: f'{x:.1%}' if isinstance(x, float) else x)
summary['vs best_simple'] = summary['vs best_simple'].apply(
    lambda x: f'{x:.1%}' if isinstance(x, float) else x)
display(summary.set_index('Modelo'))

print(f'\nResultado final del ensemble: MAE = {ens_mae:.4f}')